In [ ]:
import requests
import pandas as pd
import joblib


In [ ]:
#load model
model = joblib.load("nb_recommender.pkl")
vectorizer = joblib.load("vectorizer.pkl")

ML_DATASET_URL   = "http://localhost:8080/api/ml/dataset"
RANDOM_ART_URL   = "http://localhost:8080/api/artworks/random"

In [ ]:
def recommend_artworks(user_id, top_n=5):

    # fetching data for the user
    dataset = requests.get(ML_DATASET_URL).json()
    user_rows = [row for row in dataset if row["userId"] == user_id]

    if not user_rows:
        print(f"No data found for user {user_id}")
        return []

    prefs = user_rows[0]
    user_prefs_text = " ".join([
        prefs.get("preferredArtists", "") or "",
        prefs.get("preferredStyles", "") or "",
        prefs.get("preferredMediums", "") or "",
        prefs.get("preferredTimePeriods", "") or "",
        prefs.get("preferredMovements", "") or "",
    ])

    # Get artworks the user has already interacted with
    seen_ids = set(row["artworkId"] for row in user_rows)

    # fetch canditate artworks
    candidates = requests.get(RANDOM_ART_URL).json()

    # filter out aready seen artworks
    unseen = [a for a in candidates if a["objectID"] not in seen_ids]

    if not unseen:
        print("No new artworks to recommend — all candidates already seen.")
        return []

    rows = []
    for artwork in unseen:
        combined = " ".join([
            artwork.get("artist", "") or "",
            artwork.get("period", "") or "",
            artwork.get("culture", "") or "",
            artwork.get("medium", "") or "",
            user_prefs_text  
        ])
        rows.append({
            "objectID": artwork["objectID"],
            "title": artwork.get("title", ""),
            "artist": artwork.get("artist", ""),
            "period": artwork.get("period", ""),
            "imageUrl": artwork.get("imageUrl", ""),
            "combined_text": combined
        })

    df_candidates = pd.DataFrame(rows)

    X = vectorizer.transform(df_candidates["combined_text"])

    df_candidates["like_probability"] = model.predict_proba(X)[:, 1]

    # sort by highest probability 
    recommendations = (
        df_candidates
        .sort_values("like_probability", ascending=False)
        .head(top_n)
        [["objectID", "title", "artist","period", "imageUrl", "like_probability"]]
        .reset_index(drop=True)
    )
    return recommendations


In [ ]:
#test with a user ID
user_id_to_test = 8
results = recommend_artworks(user_id_to_test, top_n=5)
print(results)